# ResNet Continue Training — Epoch 30 → 100
### Already has Perceptual Loss — just increase TOTAL_EPOCHS

**Before running:**
1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Internet → **On**
3. Add Input → search **coco-2017-dataset** → Add
4. Change `FILE_ID` in Cell 5 (your epoch 30 checkpoint)
5. Session Type → **Save & Run All (Commit)** ← runs in background!

In [ ]:
# CELL 1 — Check GPU
import torch
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'GPU {i}   : {torch.cuda.get_device_name(i)}')
        print(f'VRAM    : {round(torch.cuda.get_device_properties(i).total_memory/1e9,1)} GB')
    print('Ready!')
else:
    print('NO GPU!')

In [ ]:
# CELL 2 — Install packages
!pip install -q scikit-image tqdm gdown
print('Packages ready!')

In [ ]:
# CELL 3 — Find COCO dataset
import os
from pathlib import Path

train_path = None
val_path   = None

print('Searching for COCO images...')
for root, dirs, files in os.walk('/kaggle/input'):
    jpgs = [f for f in files if f.endswith('.jpg')]
    if len(jpgs) > 1000:
        print(f'Found {len(jpgs):,} images at: {root}')
        if 'train' in root.lower() and train_path is None:
            train_path = root
        elif 'val' in root.lower() and val_path is None:
            val_path = root

print(f'Train: {train_path}')
print(f'Val  : {val_path}')

In [ ]:
# CELL 4 — Set dataset paths
from pathlib import Path

possible_train = [
    '/kaggle/input/coco-2017-dataset/coco2017/train2017',
    '/kaggle/input/coco-2017-dataset/train2017',
    train_path,
]
possible_val = [
    '/kaggle/input/coco-2017-dataset/coco2017/val2017',
    '/kaggle/input/coco-2017-dataset/val2017',
    val_path,
]

TRAIN_DIR = None
VAL_DIR   = None

for p in possible_train:
    if p and Path(p).exists():
        count = len(list(Path(p).glob('*.jpg')))
        if count > 100:
            TRAIN_DIR = Path(p)
            print(f'Train: {p} ({count:,} images)')
            break

for p in possible_val:
    if p and Path(p).exists():
        count = len(list(Path(p).glob('*.jpg')))
        if count > 100:
            VAL_DIR = Path(p)
            print(f'Val  : {p} ({count:,} images)')
            break

if not TRAIN_DIR:
    print('ERROR: Could not find COCO images!')

In [ ]:
# CELL 5 — Download checkpoint + loss history
# !! CHANGE THESE LINES ONLY !!
FILE_ID       = 'YOUR_GOOGLE_DRIVE_FILE_ID'  # <- generator_resnet_epoch53.pth file ID
MY_EPOCH      = 53                            # <- your current epoch number
LOSS_FILE_ID  = 'YOUR_LOSS_JSON_FILE_ID'      # <- loss_history_resnet.json file ID (or '' to skip)
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

import os, shutil
import gdown

CKPT_DIR = '/kaggle/working/ColorizeGAN_ResNet/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

# ── Download checkpoint ───────────────────────────────────────────────────
print(f'Downloading generator_resnet_epoch{MY_EPOCH}.pth ...')
gdown.download(id=FILE_ID, output='/kaggle/working/downloaded_model.pth', quiet=False)

if os.path.exists('/kaggle/working/downloaded_model.pth'):
    size = os.path.getsize('/kaggle/working/downloaded_model.pth') / (1024*1024)
    print(f'Downloaded: {size:.1f} MB')
    if size < 50:
        print('ERROR: File too small — check FILE_ID!')
    else:
        dest = f'{CKPT_DIR}/generator_resnet_epoch{MY_EPOCH}.pth'
        shutil.copy('/kaggle/working/downloaded_model.pth', dest)
        print(f'✅ Checkpoint saved: generator_resnet_epoch{MY_EPOCH}.pth')
        print(f'   Training resumes from epoch {MY_EPOCH + 1}')
else:
    print('ERROR: Download failed! Check FILE_ID and Drive share permissions')

# ── Download loss history JSON (optional but recommended) ────────────────
if LOSS_FILE_ID and LOSS_FILE_ID != 'YOUR_LOSS_JSON_FILE_ID':
    print('\nDownloading loss_history_resnet.json ...')
    gdown.download(id=LOSS_FILE_ID, output='/kaggle/working/downloaded_loss.json', quiet=False)
    if os.path.exists('/kaggle/working/downloaded_loss.json'):
        dest_json = f'{CKPT_DIR}/loss_history_resnet.json'
        shutil.copy('/kaggle/working/downloaded_loss.json', dest_json)
        import json as json_lib
        with open(dest_json) as f:
            d = json_lib.load(f)
        print(f'✅ Loss history loaded: ep.{d["epochs"][0]} to ep.{d["epochs"][-1]} ({len(d["epochs"])} epochs)')
        print(f'   This session will APPEND to existing history')
    else:
        print('WARNING: Loss JSON download failed — will start fresh log this session')
else:
    print('\nℹ️  No LOSS_FILE_ID set — loss history starts fresh this session')
    print('   (Set LOSS_FILE_ID to keep continuous loss history across sessions)')


In [ ]:
# CELL 6 — Define Models (identical to your original resnet notebook)
import torch
import torch.nn as nn
import torchvision.models as tv_models
import numpy as np
from torch.utils.data import Dataset, DataLoader
from skimage.color import rgb2lab, lab2rgb
from PIL import Image
import random

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
print('GPUs  :', torch.cuda.device_count())

class ColorizationDataset(Dataset):
    def __init__(self, root_dir, size=256):
        self.files = sorted(Path(root_dir).glob('*.jpg'))
        self.size  = size
        print(f'Dataset: {len(self.files):,} images')
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        try:
            img    = Image.open(self.files[idx]).convert('RGB')
            img    = img.resize((self.size, self.size), Image.LANCZOS)
            img_np = np.array(img, dtype=np.uint8)
            lab    = rgb2lab(img_np).astype('float32')
            L      = torch.tensor(lab[:,:,0] / 50.0 - 1.0).unsqueeze(0)
            AB     = torch.tensor(lab[:,:,1:] / 110.0).permute(2,0,1)
            return L, AB
        except:
            return self.__getitem__(random.randint(0, len(self.files)-1))

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=False):
        super().__init__()
        layers = [nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1, bias=False),
                  nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)]
        if dropout: layers.append(nn.Dropout(0.5))
        self.block = nn.Sequential(*layers)
    def forward(self, x): return self.block(x)

class Generator(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        resnet = tv_models.resnet34(weights='IMAGENET1K_V1' if pretrained else None)
        self.enc0 = nn.Sequential(nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False),
                                   resnet.bn1, resnet.relu)
        self.enc1 = resnet.maxpool
        self.enc2 = resnet.layer1
        self.enc3 = resnet.layer2
        self.enc4 = resnet.layer3
        self.enc5 = resnet.layer4
        self.bottleneck = nn.Sequential(nn.Conv2d(512, 512, 3, padding=1, bias=False),
                                         nn.BatchNorm2d(512), nn.ReLU(inplace=True))
        self.dec5 = ConvBlock(512,       512, dropout=True)
        self.dec4 = ConvBlock(512 + 256, 256, dropout=True)
        self.dec3 = ConvBlock(256 + 128, 128)
        self.dec2 = ConvBlock(128 + 64,  64)
        self.dec1 = ConvBlock(64  + 64,  64)
        self.out  = nn.Sequential(nn.Conv2d(64, 2, kernel_size=3, padding=1), nn.Tanh())
        for param in self.enc0.parameters(): param.requires_grad = False
        for param in self.enc2.parameters(): param.requires_grad = False
    def forward(self, x):
        e0 = self.enc0(x); e1 = self.enc1(e0); e2 = self.enc2(e1)
        e3 = self.enc3(e2); e4 = self.enc4(e3); e5 = self.enc5(e4)
        bn = self.bottleneck(e5)
        d5 = self.dec5(bn)
        d4 = self.dec4(torch.cat([d5, e4], 1))
        d3 = self.dec3(torch.cat([d4, e3], 1))
        d2 = self.dec2(torch.cat([d3, e2], 1))
        d1 = self.dec1(torch.cat([d2, e0], 1))
        return self.out(d1)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3,   64,  4, 2, 1), nn.LeakyReLU(0.2),
            nn.Conv2d(64,  128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2),
            nn.Conv2d(256, 512, 4, 1, 1), nn.BatchNorm2d(512), nn.LeakyReLU(0.2),
            nn.Conv2d(512, 1,   4, 1, 1)
        )
    def forward(self, L, AB): return self.model(torch.cat([L, AB], 1))

G = Generator(pretrained=True).to(DEVICE)
D = Discriminator().to(DEVICE)

if torch.cuda.device_count() > 1:
    print(f'Using {torch.cuda.device_count()} GPUs!')
    G = nn.DataParallel(G)
    D = nn.DataParallel(D)

print(f'Generator     : {sum(p.numel() for p in G.parameters())/1e6:.1f}M params')
print(f'Discriminator : {sum(p.numel() for p in D.parameters())/1e6:.1f}M params')
print('Ready!')

In [ ]:
# CELL 7 — Train ResNet (epoch 31 → 100) with Real Loss Logging
import time, glob, os, shutil, json as json_lib
import matplotlib.pyplot as plt
from tqdm import tqdm

# ── Config ────────────────────────────────────────────────
TOTAL_EPOCHS = 100
LR           = 1e-4
BATCH_SIZE   = 8
LAMBDA_L1    = 100
LAMBDA_PERC  = 10
SAVE_EVERY   = 1

train_ds = ColorizationDataset(TRAIN_DIR)
val_ds   = ColorizationDataset(VAL_DIR)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE,
                      shuffle=True,  num_workers=4, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=8,
                      shuffle=False, num_workers=4, pin_memory=True)
print(f'Train batches : {len(train_dl):,}')

opt_G = torch.optim.Adam(
    filter(lambda p: p.requires_grad, G.parameters()),
    lr=LR, betas=(0.5, 0.999)
)
opt_D   = torch.optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.999))
sched_G = torch.optim.lr_scheduler.StepLR(opt_G, step_size=25, gamma=0.5)
sched_D = torch.optim.lr_scheduler.StepLR(opt_D, step_size=25, gamma=0.5)
bce = nn.BCEWithLogitsLoss()
l1  = nn.L1Loss()

# VGG Perceptual Loss (same as original)
vgg = tv_models.vgg16(weights='IMAGENET1K_V1').features[:16].to(DEVICE).eval()
for p in vgg.parameters(): p.requires_grad = False
print('VGG perceptual loss ready!')

def perceptual_loss(pred_ab, real_ab, L):
    pred_rgb = torch.cat([(L + 1) * 0.5,
                           pred_ab[:, 0:1] / 2 + 0.5,
                           pred_ab[:, 1:2] / 2 + 0.5], dim=1)
    real_rgb = torch.cat([(L + 1) * 0.5,
                           real_ab[:, 0:1] / 2 + 0.5,
                           real_ab[:, 1:2] / 2 + 0.5], dim=1)
    return nn.functional.l1_loss(vgg(pred_rgb), vgg(real_rgb).detach())

# ── Loss log path ──────────────────────────────────────────
LOSS_LOG = f'{CKPT_DIR}/loss_history_resnet.json'

def save_checkpoint(epoch, is_final=False):
    import subprocess
    name = 'generator_resnet_final.pth' if is_final else f'generator_resnet_epoch{epoch}.pth'
    path = f'{CKPT_DIR}/{name}'
    state = G.module.state_dict() if hasattr(G, 'module') else G.state_dict()
    torch.save(state, path)
    subprocess.run(['sync'], capture_output=True)
    if os.path.exists(path):
        size = os.path.getsize(path) / (1024*1024)
        label = 'FINAL' if is_final else f'epoch{epoch}'
        print(f'   SAVED -> generator_resnet_{label}.pth ({size:.1f} MB)')
        shutil.copy(path, f'/kaggle/working/generator_resnet_{label}.pth')
        # Auto-delete old checkpoints — keep only last 2
        old = sorted(glob.glob(f'{CKPT_DIR}/generator_resnet_epoch*.pth'))
        if len(old) > 2:
            for old_f in old[:-2]:
                os.remove(old_f)
                print(f'   Deleted old: {os.path.basename(old_f)}')
        return True
    return False

# ── Auto Resume ────────────────────────────────────────────
existing    = sorted(glob.glob(f'{CKPT_DIR}/generator_resnet_epoch*.pth'))
START_EPOCH = 1
if existing:
    latest = existing[-1]
    try:
        epoch_num   = int(os.path.basename(latest).replace('generator_resnet_epoch','').replace('.pth',''))
        START_EPOCH = epoch_num + 1
        state = torch.load(latest, map_location=DEVICE)
        if hasattr(G, 'module'): G.module.load_state_dict(state)
        else: G.load_state_dict(state)
        print(f'AUTO RESUMED from: {os.path.basename(latest)}')
        print(f'Continuing from epoch {START_EPOCH}')
    except Exception as e:
        print(f'Resume failed: {e}')
        START_EPOCH = 1
else:
    print('Starting fresh!')

# ── Load existing loss log (important for resume!) ─────────
if os.path.exists(LOSS_LOG) and START_EPOCH > 1:
    with open(LOSS_LOG) as f:
        history = json_lib.load(f)
    print(f'Loss log loaded — {len(history["epochs"])} epochs already recorded')
else:
    history = {'epochs': [], 'G': [], 'D': [], 'Perc': []}
    print('Starting fresh loss log')

print(f'Training epoch {START_EPOCH} -> {TOTAL_EPOCHS}')
print('='*55)

for epoch in range(START_EPOCH, TOTAL_EPOCHS + 1):
    G.train(); D.train()
    g_losses, d_losses, p_losses = [], [], []
    t0 = time.time()

    for L, real_AB in tqdm(train_dl, desc=f'Epoch {epoch}/{TOTAL_EPOCHS}', leave=False):
        L, real_AB = L.to(DEVICE), real_AB.to(DEVICE)

        # Train Discriminator
        fake_AB = G(L)
        D_real  = D(L, real_AB)
        D_fake  = D(L, fake_AB.detach())
        loss_D  = (bce(D_real, torch.ones_like(D_real)) +
                   bce(D_fake, torch.zeros_like(D_fake))) * 0.5
        opt_D.zero_grad(); loss_D.backward(); opt_D.step()

        # Train Generator
        fake_AB  = G(L)
        D_fake   = D(L, fake_AB)
        loss_GAN = bce(D_fake, torch.ones_like(D_fake))
        loss_L1  = l1(fake_AB, real_AB) * LAMBDA_L1
        loss_P   = perceptual_loss(fake_AB, real_AB, L) * LAMBDA_PERC
        loss_G   = loss_GAN + loss_L1 + loss_P
        opt_G.zero_grad(); loss_G.backward(); opt_G.step()

        g_losses.append(loss_G.item())
        d_losses.append(loss_D.item())
        p_losses.append(loss_P.item())

    avg_G   = float(np.mean(g_losses))
    avg_D   = float(np.mean(d_losses))
    avg_P   = float(np.mean(p_losses))
    elapsed = (time.time() - t0) / 60

    # ── Append to real loss log ────────────────────────────
    history['epochs'].append(epoch)
    history['G'].append(avg_G)
    history['D'].append(avg_D)
    history['Perc'].append(avg_P)

    # Save loss log every epoch
    with open(LOSS_LOG, 'w') as f:
        json_lib.dump(history, f, indent=2)
    shutil.copy(LOSS_LOG, '/kaggle/working/loss_history_resnet.json')

    sched_G.step()
    sched_D.step()

    print(f'Epoch {epoch:3d}/{TOTAL_EPOCHS} | '
          f'G: {avg_G:.4f} | D: {avg_D:.4f} | '
          f'Perc: {avg_P:.4f} | {elapsed:.1f} min | '
          f'LR: {sched_G.get_last_lr()[0]:.2e}')

    if epoch % SAVE_EVERY == 0:
        save_checkpoint(epoch)

save_checkpoint(TOTAL_EPOCHS, is_final=True)

# Plot real loss curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history['epochs'], history['G'],    color='#2563eb', linewidth=2); axes[0].set_title('Generator Loss');    axes[0].grid(alpha=0.3)
axes[1].plot(history['epochs'], history['D'],    color='#dc2626', linewidth=2); axes[1].set_title('Discriminator Loss'); axes[1].grid(alpha=0.3)
axes[2].plot(history['epochs'], history['Perc'], color='#16a34a', linewidth=2); axes[2].set_title('Perceptual Loss');    axes[2].grid(alpha=0.3)
for ax in axes: ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
plt.suptitle(f'ResNet Real Training Loss — ep.{history["epochs"][0]} to {history["epochs"][-1]}', fontsize=13)
plt.tight_layout()
plt.savefig('/kaggle/working/resnet_training_loss.png', dpi=150, bbox_inches='tight')
plt.show()
print('Done! Download loss_history_resnet.json and generator_resnet_epochXXX.pth from Output tab')

In [ ]:
# CELL 8 — Visualize Results
import matplotlib.pyplot as plt

G.eval()
L_batch, AB_batch = next(iter(val_dl))
L_batch = L_batch.to(DEVICE)

with torch.no_grad():
    fake_AB = G(L_batch)

def lab_to_rgb(L, AB):
    L_  = (L.squeeze().cpu().numpy() + 1.0) * 50.0
    AB_ = AB.squeeze().permute(1,2,0).cpu().numpy() * 110.0
    lab = np.concatenate([L_[:,:,None], AB_], axis=2)
    return (lab2rgb(lab)*255).clip(0,255).astype(np.uint8)

n = 4
fig, axes = plt.subplots(3, n, figsize=(16, 10))
for i in range(n):
    gray = ((L_batch[i].squeeze().cpu().numpy()+1)*127.5).astype(np.uint8)
    pred = lab_to_rgb(L_batch[i], fake_AB[i])
    gt   = lab_to_rgb(L_batch[i], AB_batch[i])
    for row, (ax, img) in enumerate(zip(axes[:,i], [gray, pred, gt])):
        ax.imshow(img, cmap='gray' if row==0 else None)
        ax.axis('off')
        if i == 0: ax.set_title(['Grayscale','ResNet Output','Ground Truth'][row],
                                  fontsize=11, fontweight='bold')
plt.suptitle('ResNet Results — Epoch 100', fontsize=14)
plt.tight_layout()
plt.savefig('/kaggle/working/resnet_results.png', dpi=150)
plt.show()
print('Done!')

In [ ]:
# CELL 9 — Files ready to download
import glob, os, json as json_lib

print('=' * 50)
print('CHECKPOINTS:')
print('-' * 50)
for c in sorted(glob.glob('/kaggle/working/*.pth')):
    size = os.path.getsize(c) / (1024*1024)
    print(f'  {os.path.basename(c):45s} {size:.1f} MB')

print('\nLOSS HISTORY:')
print('-' * 50)
for j in sorted(glob.glob('/kaggle/working/*.json')):
    with open(j) as f:
        d = json_lib.load(f)
    ep = d.get('epochs', [])
    if ep:
        print(f'  {os.path.basename(j):45s} ep.{ep[0]} to ep.{ep[-1]} ({len(ep)} epochs)')

print('=' * 50)
print('DOWNLOAD BOTH FILES each session:')
print('  Right sidebar → Output tab → click each file')
print('  Place in: checkpoints/ folder on your laptop')
print('  Upload loss JSON to Google Drive → get FILE_ID for next session')
